# 向量库记忆（Vector Store Memory）

> **把每一轮对话嵌入成向量，在推理时检索最相关的 top-K 条历史轮次。这就是把语义搜索（semantic search）引入对话记忆。**

想象一座没有目录的图书馆。想找任何一本书，你都得一排排书架走过去。再想象一位图书管理员，她一眼就知道哪三本书最能回答你的问题。向量库记忆就是给你的智能体（agent）配上这样一位管理员。它找上下文不再靠*什么时候说的*，而是靠*说的是什么*。

在[前面的短期记忆 notebook](../03_summary_memory/summary_memory.ipynb) 里，我们探讨的是基于新近度（recency）的策略：保留最后 *k* 条消息（滑动窗口），或者把旧轮次压缩成摘要。两者依赖的都是内容**何时**被说出，而不是**说了什么**。

**向量库记忆**走的是另一条路。它把每一轮对话都转换成一个稠密嵌入向量（embedding，一串刻画语义的数字），存进向量数据库。智能体需要上下文时，系统把当前查询嵌入成向量，用余弦相似度（cosine similarity）和*所有*已存轮次比较，返回语义上最相关的 top-*K* 个片段，不管它们出现在什么时候。

这和检索增强生成（RAG，Retrieval-Augmented Generation，指在生成回复前先取回相关文本）是同一个思路。这里我们把它用在智能体自己的对话历史上，而不是外部文档上。

**回报是：**用户在第 47 轮问起某个事实时，智能体能召回第 3 轮说过的内容。换成滑动窗口，这个事实早就被丢掉了。

**读完本 notebook，你将理解：**
- 如何用 OpenAI 嵌入和 ChromaDB 从零构建向量库记忆。
- 检索流程：嵌入、存储、查询、注入提示词。
- 一个 50 轮对照实验，证明对非顺序性问题，语义召回远胜滑动窗口。
- 相关权衡：检索质量、成本、延迟，以及什么时候该把它和其他记忆策略组合使用。

## 核心概念

- **嵌入模型（embedding model）**：把文本映射成定长向量的模型（例如 OpenAI 的 `text-embedding-3-small`），向量刻画文本的语义。相似的文本产生相似的向量。
- **向量数据库（vector database）**：为高维向量的近似最近邻（ANN）搜索而优化的存储。ANN 的意思是「在百万级数据里也能快速找到最接近的结果」。本 notebook 用的是 **ChromaDB**（内存模式）。
- **余弦相似度（cosine similarity）**：我们用来比较向量的距离度量。余弦相似度越高，语义越相关。
- **Top-K 检索**：对给定查询返回最相似的 *K* 个已存向量。*K* 控制查全率和成本之间的权衡：K 小则便宜，但可能漏掉内容；K 大则覆盖更全，但消耗更多 token。
- **分块策略（chunk strategy）**：嵌入前怎么切分对话：按单条消息、按用户-助手成对消息，或按滑动窗口。我们嵌入**用户-助手成对消息**，让检索到的内容更完整连贯。
- **元数据（metadata）**：附在每个已存向量上的时间戳、轮次编号等属性，可以用来做混合过滤，比如「相关*而且*较新」。
- **上下文注入（context injection）**：把检索到的记忆格式化后加到提示词前面，供 LLM 参考。

## 架构

<p align="center">
  <img src="../../images/diagrams/06_vector_store_memory.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    subgraph Ingestion["Ingestion (after each turn)"]
        A["User message +\nAssistant reply"] --> B["Embedding\nModel"]
        B --> C["Vector +\nMetadata"]
        C --> D[("ChromaDB\n(in-memory)")]
    end

    subgraph Retrieval["Retrieval (before each LLM call)"]
        E["New user\nquery"] --> F["Embedding\nModel"]
        F --> G["Cosine\nSimilarity Search"]
        D --> G
        G --> H["Top-K\nRelevant Turns"]
    end

    subgraph Generation["Generation"]
        H --> I["Build Prompt:\nsystem + retrieved\nmemories + recent buffer"]
        I --> J["LLM\n(Claude)"]
        J --> K["Response"]
    end

    style D fill:#4f46e5,color:#fff
    style J fill:#059669,color:#fff
```

</details>

In [ ]:
# 安装所需依赖（只需运行一次）
%pip install -q chromadb python-dotenv matplotlib numpy deepagents langchain-openai

加载环境变量并创建 API 客户端。我们需要两把钥匙：Anthropic 的 key（给 Claude 用）和 OpenAI 的 key（给把文本转换成向量的嵌入模型用）。

In [ ]:
import os
import chromadb

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def usage_of(reply, prompt_messages: list[dict]) -> dict:
    """读取一轮对话的 token 用量；端点未返回时用 tiktoken 估算。"""
    import tiktoken

    usage = getattr(reply, "usage_metadata", None) or {}
    encoder = tiktoken.get_encoding("cl100k_base")
    prompt_text = "".join(str(m.get("content", "")) for m in prompt_messages)
    reply_text = to_text(reply.content)
    return {
        "input_tokens": int(usage.get("input_tokens") or len(encoder.encode(prompt_text))),
        "output_tokens": int(usage.get("output_tokens") or len(encoder.encode(reply_text))),
    }
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 核心实现

我们的 `VectorStoreMemory` 有三项职责：

1. 每轮用户-助手对话发生后，**嵌入并存储**这次对话。
2. 构建下一次提示词时，**检索**最相关的 top-K 条历史对话。
3. **构建提示词**：把检索到的记忆和一小段近期消息缓冲组合起来。

我们使用：
- 用 **OpenAI `text-embedding-3-small`** 做嵌入（1536 维，便宜又快）。
- 用 **ChromaDB** 内存集合做向量存储和搜索。
- 用 **Anthropic Claude** 做对话模型。

In [ ]:
class VectorStoreMemory:
    """向量库记忆：检索语义相关的历史轮次。"""

    def __init__(
        self,
        model,
        top_k: int = 5,
        recent_buffer_size: int = 4,
        system_prompt: str | None = None,
        collection_name: str = "conversation_memory",
    ):
        self.model = model
        self.top_k = top_k
        self.recent_buffer_size = recent_buffer_size
        self.system_prompt = system_prompt

        # ChromaDB 内存集合
        self.chroma = chromadb.Client()
        self.collection = self.chroma.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

        # 状态
        self.turn_count = 0
        self.recent_buffer: list[dict] = []  # 最近 N 条消息（原文）
        self.full_history: list[dict] = []   # 完整历史，用于分析

        # 追踪记录
        self.turn_token_usage: list[dict] = []
        self.retrieval_log: list[dict] = []  # 每轮检索到了什么


接下来添加嵌入和存储方法。`_embed` 方法调用 OpenAI 的嵌入 API，把文本转换成向量。`_store_exchange` 方法把用户消息和助手回复拼在一起，对这一对做嵌入，然后连同元数据写入 ChromaDB。

In [ ]:
# -- 嵌入 -------------------------------------------------------
def _embed(self, text: str) -> list[float]:
    """获取一段文本的嵌入向量。"""
    return embed_texts([text])[0]


# -- 存储 ------------------------------------------------------------
def _store_exchange(self, user_msg: str, assistant_msg: str) -> None:
    """嵌入并存储一轮用户-助手对话。"""
    self.turn_count += 1
    # 拼接用户和助手消息，让嵌入包含更完整的信息
    combined = f"User: {user_msg}\nAssistant: {assistant_msg}"
    embedding = self._embed(combined)

    self.collection.add(
        ids=[f"turn-{self.turn_count}"],
        embeddings=[embedding],
        documents=[combined],
        metadatas=[{
            "turn": self.turn_count,
            "user_msg": user_msg,
            "assistant_msg": assistant_msg,
        }],
    )


VectorStoreMemory._embed = _embed
VectorStoreMemory._store_exchange = _store_exchange

print("✓ embedding and storage methods attached")


然后添加检索方法。你发送新消息时，记忆先把查询嵌入成向量，再到 ChromaDB 里搜索最接近的已存对话，同时把这些检索到的记忆格式化成系统提示词，作为 LLM 的上下文。

In [ ]:
# -- 检索 ---------------------------------------------------------
def _retrieve(self, query: str, top_k: int | None = None) -> list[dict]:
    """检索最相关的 top-K 条历史对话。"""
    k = top_k or self.top_k
    if self.turn_count == 0:
        return []

    query_embedding = self._embed(query)
    results = self.collection.query(
        query_embeddings=[query_embedding],
        n_results=min(k, self.turn_count),
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "turn": results["metadatas"][0][i]["turn"],
            "document": results["documents"][0][i],
            "distance": results["distances"][0][i] if results["distances"] else None,
        })
    return retrieved


# -- 构建提示词 -----------------------------------------------------
def _build_system_prompt(self, retrieved: list[dict]) -> str:
    """把系统提示词和检索到的记忆组合起来。"""
    parts = []
    if self.system_prompt:
        parts.append(self.system_prompt)

    if retrieved:
        memory_text = "\n\n".join(
            f"[Turn {r['turn']}] {r['document']}" for r in retrieved
        )
        parts.append(
            f"The following are relevant past exchanges from this conversation:\n\n"
            f"{memory_text}\n\n"
            f"Use these memories to inform your response when relevant."
        )
    return "\n\n".join(parts) if parts else "You are a helpful assistant."


VectorStoreMemory._retrieve = _retrieve
VectorStoreMemory._build_system_prompt = _build_system_prompt

print("✓ retrieval and prompt methods attached")


`chat` 方法把所有环节串起来：检索相关的历史轮次、构建提示词、调用 Claude、存储这轮新对话、更新近期缓冲。每收到一条用户消息，你调用的就是这个主入口。

In [ ]:
# -- 对话 -------------------------------------------------------------
def _chat(self, user_input: str) -> str:
    """发送消息，检索相关上下文，并获取回复。"""
    # 第 1 步：检索相关的历史轮次
    retrieved = self._retrieve(user_input)
    self.retrieval_log.append({
        "turn": self.turn_count + 1,
        "query": user_input,
        "retrieved_turns": [r["turn"] for r in retrieved],
    })

    # 第 2 步：构建消息列表（近期缓冲 + 新消息）
    messages = list(self.recent_buffer) + [{"role": "user", "content": user_input}]

    # 第 3 步：调用 deepagents（系统提示词带检索上下文，每轮重建）
    agent = create_deep_agent(
        model=self.model,
        system_prompt=self._build_system_prompt(retrieved),
    )
    result = agent.invoke({"messages": messages})
    reply = result["messages"][-1]
    assistant_text = to_text(reply.content)

    # 第 4 步：把这轮对话存入向量数据库
    self._store_exchange(user_input, assistant_text)

    # 第 5 步：更新近期缓冲
    self.recent_buffer.append({"role": "user", "content": user_input})
    self.recent_buffer.append({"role": "assistant", "content": assistant_text})
    while len(self.recent_buffer) > self.recent_buffer_size:
        self.recent_buffer.pop(0)

    # 完整历史，用于分析
    self.full_history.append({"role": "user", "content": user_input})
    self.full_history.append({"role": "assistant", "content": assistant_text})

    # 记录 token 用量
    usage = usage_of(reply, messages)
    self.turn_token_usage.append({
        "turn": self.turn_count,
        "input_tokens": usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
        "retrieved_count": len(retrieved),
    })

    return assistant_text


VectorStoreMemory.chat = _chat

print("✓ VectorStoreMemory.chat attached")


最后添加几个检查用的辅助方法。`search` 方法让你直接查询向量库，方便调试。`stats` 方法报告存储了多少轮对话和多少个向量。

In [ ]:
# -- 检查辅助 -------------------------------------------------------
def _search(self, query: str, top_k: int = 5) -> list[dict]:
    """公开的搜索方法，用于调试记忆里存了什么。"""
    return self._retrieve(query, top_k)


def _stats(self) -> dict:
    return {
        "total_turns": self.turn_count,
        "vectors_stored": self.collection.count(),
        "recent_buffer_size": len(self.recent_buffer),
    }


def _repr(self) -> str:
    return (
        f"VectorStoreMemory(turns={self.turn_count}, "
        f"vectors={self.collection.count()}, "
        f"top_k={self.top_k})"
    )


VectorStoreMemory.search = _search
VectorStoreMemory.stats = _stats
VectorStoreMemory.__repr__ = _repr

print("✓ VectorStoreMemory helper methods attached")


## 使用示例：语义召回实战

我们先在多个轮次里埋下几条事实，再打乱顺序去问。即使相关内容出现在很多条消息之前，向量库也应该能检索到对应的轮次。

In [ ]:
mem = VectorStoreMemory(
    model=llm,
    top_k=3,
    recent_buffer_size=4,
    system_prompt="You are a concise assistant. Reply in 1-2 sentences.",
    collection_name="usage_demo",
)

messages = [
    "My name is Priya and I'm a data scientist at Spotify.",
    "I adopted a rescue greyhound named Zephyr last month.",
    "I'm learning Rust to build a music recommendation engine.",
    "My favorite cuisine is Ethiopian -- I love injera with misir wot.",
    "What pet do I have?",              # 应检索到第 2 轮
    "What programming language am I learning?",  # 应检索到第 3 轮
]

for msg in messages:
    print(f"\U0001f464 User:  {msg}")
    reply = mem.chat(msg)
    print(f"\U0001f916 Agent: {reply}")
    print()

print(f"\n\U0001f4ca Stats: {mem.stats()}")

我们来看看向量库为每个查询检索到了什么。检索日志显示每条用户消息命中了哪些已存轮次，方便你确认语义搜索找对了上下文。

In [ ]:
# 看看向量库为每个查询检索到了什么
print("=== Retrieval Log ===\n")
for entry in mem.retrieval_log:
    if entry["retrieved_turns"]:
        print(f"Turn {entry['turn']}: \"{entry['query'][:50]}...\"" 
              if len(entry["query"]) > 50 
              else f"Turn {entry['turn']}: \"{entry['query']}\"" )
        print(f"  \u2192 Retrieved turns: {entry['retrieved_turns']}")
        print()

你也可以直接搜索这份记忆，就像在对话上跑一个语义搜索引擎。这对调试很有用：传入一个查询，看看哪些已存对话在语义上最接近。

In [ ]:
# 直接搜索记忆：相当于在对话上跑一个语义搜索引擎
results = mem.search("food preferences", top_k=3)
print("\U0001f50d Search: 'food preferences'\n")
for r in results:
    print(f"  Turn {r['turn']} (distance: {r['distance']:.4f}):")
    print(f"    {r['document'][:100]}...")
    print()

## 实验：50 轮对话，向量库 vs 滑动窗口

这是本 notebook 的关键实验。我们会：

1. 生成一段 **50 轮合成对话**，在已知轮次埋下特定事实。
2. 50 轮结束后，提出引用早期事实的**召回问题**。
3. 把同样的问题分别喂给**向量库记忆**（top-K=5）和**滑动窗口**（K=10 条消息）。
4. 比较召回准确率：哪个系统记得更多？

假设是：向量库记忆能召回对话中任意位置的事实，滑动窗口只能召回最后约 5 轮里的事实。

In [ ]:
# 50 轮合成对话：在特定轮次埋入事实，其余轮次用填充消息隔开。
# 每个条目是 (轮次编号, 用户消息)。
# "Fact" 轮次埋入可检索的具体信息。
# "Filler" 轮次是普通问题，作用是把事实挤出滑动窗口。

PLANTED_FACTS = {
    1:  ("My name is Jordan Rivera and I'm 34 years old.", "jordan rivera", "name"),
    3:  ("I work as a robotics engineer at Boston Dynamics.", "boston dynamics", "employer"),
    5:  ("My blood type is AB negative -- it's pretty rare.", "ab negative", "blood type"),
    8:  ("I'm deathly allergic to shellfish -- I carry an EpiPen.", "shellfish", "allergy"),
    12: ("My childhood dog was a beagle named Biscuit.", "biscuit", "childhood pet"),
    16: ("I ran the Boston Marathon last year in 3 hours 42 minutes.", "3 hours 42", "marathon"),
    20: ("My Wi-Fi password at home is 'correct-horse-battery-staple'.", "correct-horse", "password"),
    25: ("I have a twin sister named Avery who lives in Barcelona.", "avery", "sister"),
    30: ("My favorite book is 'Godel, Escher, Bach' by Hofstadter.", "godel", "book"),
    35: ("I proposed to my partner at the top of Machu Picchu.", "machu picchu", "proposal"),
    40: ("My car is a 2019 Subaru Outback in dark green.", "subaru outback", "car"),
    45: ("I donated a kidney to my college roommate in 2021.", "kidney", "donation"),
}



我们用填充消息来填满埋点事实之间空余的轮次。这些是普通问题，作用是把埋下的事实挤出滑动窗口。在真实对话里，它们就是重要时刻之间的闲聊。

In [ ]:
FILLER_MESSAGES = [
    "What's the weather like in Boston today?",
    "Can you explain how neural networks work?",
    "Tell me a fun fact about octopuses.",
    "What's the capital of Mongolia?",
    "How do you make sourdough bread?",
    "What's the difference between TCP and UDP?",
    "Tell me about the James Webb Space Telescope.",
    "How does CRISPR gene editing work?",
    "What are the rules of cricket?",
    "Explain the Monty Hall problem.",
    "What's the deepest cave in the world?",
    "How do tides work?",
    "Tell me about the history of the internet.",
    "What causes thunder?",
    "How do airplanes stay in the air?",
    "What's the largest desert on Earth?",
    "Explain blockchain in simple terms.",
    "What are the Northern Lights?",
    "How does a microwave oven work?",
    "What's the fastest land animal?",
    "Tell me about the Voynich manuscript.",
    "How do dolphins sleep?",
    "What causes earthquakes?",
    "How does GPS navigation work?",
    "What's the oldest known living tree?",
    "Explain the placebo effect.",
    "How do noise-canceling headphones work?",
    "What's the tallest waterfall in the world?",
    "Tell me about the Fermi Paradox.",
    "How does a refrigerator work?",
    "What causes the seasons?",
    "How does a touchscreen work?",
    "Tell me about the deep ocean.",
    "What is dark matter?",
    "How do homing pigeons navigate?",
    "What causes a rainbow?",
    "How do vaccines trigger immunity?",
    "What's the most spoken language in the world?",
]



现在我们用埋点事实和填充消息拼出 50 轮对话，同时定义召回问题。每个问题针对一条特定事实，并附带一个我们会在答案里检查的关键词。

In [ ]:
# 构建 50 轮对话
conversation_50 = []
filler_idx = 0
for turn in range(1, 51):
    if turn in PLANTED_FACTS:
        msg = PLANTED_FACTS[turn][0]
    else:
        msg = FILLER_MESSAGES[filler_idx % len(FILLER_MESSAGES)]
        filler_idx += 1
    conversation_50.append((turn, msg))

print(f"Built synthetic conversation: {len(conversation_50)} turns")
print(f"Planted facts at turns: {sorted(PLANTED_FACTS.keys())}")

# 召回问题：每个问题对应一条埋入的事实
RECALL_QUESTIONS = [
    ("What is my full name and age?",           "jordan rivera",    1),
    ("Where do I work?",                        "boston dynamics",   3),
    ("What's my blood type?",                   "ab negative",      5),
    ("What food allergy do I have?",            "shellfish",        8),
    ("What was my childhood dog's name?",       "biscuit",          12),
    ("What was my marathon time?",              "3 hours 42",       16),
    ("What's my home Wi-Fi password?",          "correct-horse",    20),
    ("Do I have any siblings?",                 "avery",            25),
    ("What's my favorite book?",                "godel",            30),
    ("Where did I propose to my partner?",      "machu picchu",     35),
    ("What car do I drive?",                    "subaru",           40),
    ("Have I ever donated an organ?",           "kidney",           45),
]

print(f"Recall questions: {len(RECALL_QUESTIONS)}")

我们把全部 50 轮喂给向量库记忆。每一轮都会被嵌入并存储。跑完后，记忆里已有 50 个向量，随时可以检索。

In [ ]:
# -- 把 50 轮对话跑一遍向量库记忆 --

vec_mem = VectorStoreMemory(
    model=llm,
    top_k=5,
    recent_buffer_size=4,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
    collection_name="experiment_vec",
)

print("Running 50-turn conversation through Vector Store Memory...")
for turn_num, msg in conversation_50:
    vec_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/50 done")

print(f"\n\u2713 Done. Vectors stored: {vec_mem.collection.count()}")

现在测试召回。对 12 条埋入的事实逐条提问，检查答案里是否包含预期的关键词。同时记录向量库检索到了哪些轮次，看它有没有找对来源。

In [ ]:
# -- 召回测试：向量库记忆 --

print("=== Vector Store Memory \u2014 Recall Test ===\n")
vec_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    # Check what would be retrieved
    retrieved = vec_mem.search(question, top_k=5)
    retrieved_turns = [r["turn"] for r in retrieved]

    answer = vec_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    vec_results.append({
        "question": question,
        "keyword": keyword,
        "planted_at": planted_at,
        "recalled": recalled,
        "retrieved_turns": retrieved_turns,
        "answer": answer,
    })

    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (planted turn {planted_at:2d}) {question}")
    print(f"    Retrieved turns: {retrieved_turns}")
    print(f"    Answer: {answer[:100]}")
    print()

vec_score = sum(1 for r in vec_results if r["recalled"])
print(f"Vector Store Memory score: {vec_score}/{len(RECALL_QUESTIONS)}")

### 基线：滑动窗口记忆（K=10）

作为对照，我们把同样的 50 轮对话跑一遍滑动窗口，它只保留最后 10 条消息（5 组用户-助手对话）。这是公平的比较：每次调用 LLM 时，两个系统用的上下文量大致相同。

In [ ]:
class SlidingWindowBaseline:
    """滑动窗口：只保留最近 K 条消息。"""

    def __init__(self, model, window_size: int = 10, system_prompt: str | None = None):
        self.agent = create_deep_agent(model=model, system_prompt=system_prompt)
        self.window_size = window_size
        self.messages: list[dict] = []

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})
        window = self.messages[-self.window_size:]

        result = self.agent.invoke({"messages": list(window)})
        assistant_text = to_text(result["messages"][-1].content)
        self.messages.append({"role": "assistant", "content": assistant_text})
        return assistant_text


# -- 让 30 轮对话跑一遍滑动窗口 --

sw_mem = SlidingWindowBaseline(
    model=llm,
    window_size=10,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 30-turn conversation through Sliding Window Memory...")
for turn_num, msg in conversation_30:
    sw_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/30 done")

print(f"\n✓ Done. Window keeps last {sw_mem.window_size} messages.")


我们把同样的 50 轮对话跑一遍滑动窗口基线。窗口只保留最后 10 条消息（5 组对话），更早的事实就此离开上下文。

In [ ]:
# -- 把同样的 50 轮对话跑一遍滑动窗口 --

sw_mem = SlidingWindowMemory(
    window_size=10,
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 50-turn conversation through Sliding Window Memory...")
for turn_num, msg in conversation_50:
    sw_mem.chat(msg)
    if turn_num % 10 == 0:
        print(f"  Turn {turn_num}/50 done")

print(f"\n\u2713 Done. Window keeps last {sw_mem.window_size} messages.")

我们用同样的召回问题去测滑动窗口。它只能看到最后 10 条消息，所以面对早已离开窗口的早期事实，它应该答不上来。

In [ ]:
# -- 召回测试：滑动窗口记忆 --

print("=== Sliding Window Memory \u2014 Recall Test ===\n")
sw_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
    answer = sw_mem.chat(question)
    recalled = keyword.lower() in answer.lower()
    sw_results.append({
        "question": question,
        "keyword": keyword,
        "planted_at": planted_at,
        "recalled": recalled,
        "answer": answer,
    })

    status = "\u2713" if recalled else "\u2717"
    print(f"  {status} (planted turn {planted_at:2d}) {question}")
    print(f"    Answer: {answer[:100]}")
    print()

sw_score = sum(1 for r in sw_results if r["recalled"])
print(f"Sliding Window Memory score: {sw_score}/{len(RECALL_QUESTIONS)}")

我们把结果画出来。面板 1 展示总召回得分，面板 2 按每条事实的「年龄」（埋入的轮次）拆解召回情况。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -- 面板 1：并排比较召回得分 --
labels = ["Vector Store\nMemory", "Sliding Window\n(K=10)"]
scores = [
    sum(1 for r in vec_results if r["recalled"]),
    sum(1 for r in sw_results if r["recalled"]),
]
colors = ["#4f46e5", "#ef4444"]
bars = axes[0].bar(labels, scores, color=colors, width=0.5, alpha=0.85)
axes[0].set_ylabel("Facts Recalled")
axes[0].set_ylim(0, len(RECALL_QUESTIONS) + 1)
axes[0].set_title("Total Facts Recalled (out of 12)")
axes[0].axhline(y=len(RECALL_QUESTIONS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(score), ha="center", fontweight="bold", fontsize=14)

# -- 面板 2：按事实年龄（埋入轮次）拆解召回 --
fact_turns = [r["planted_at"] for r in vec_results]
vec_recalled = [1 if r["recalled"] else 0 for r in vec_results]
sw_recalled = [1 if r["recalled"] else 0 for r in sw_results]

x = np.arange(len(fact_turns))
width = 0.35
axes[1].bar(x - width/2, vec_recalled, width, label="Vector Store", color="#4f46e5", alpha=0.85)
axes[1].bar(x + width/2, sw_recalled, width, label="Sliding Window", color="#ef4444", alpha=0.85)
axes[1].set_xlabel("Fact Planted at Turn #")
axes[1].set_ylabel("Recalled? (1=Yes, 0=No)")
axes[1].set_title("Recall by Fact Age")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in fact_turns], fontsize=8)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["No", "Yes"])
axes[1].legend()



面板 3 检查检索精度。对每个召回问题，向量库有没有检索到事实最初埋入的那个轮次？绿点表示命中，橙色叉表示正确的轮次没进 top-K 结果。

In [ ]:
# -- 面板 3：向量库检索到了哪些轮次？ --
retrieved_hits = []
retrieved_misses = []
for r in vec_results:
    planted = r["planted_at"]
    retrieved = r.get("retrieved_turns", [])
    if planted in retrieved:
        retrieved_hits.append(planted)
    else:
        retrieved_misses.append(planted)

axes[2].scatter(retrieved_hits, [1]*len(retrieved_hits), color="#22c55e",
                s=120, marker="o", label="Retrieved correct turn", zorder=3)
axes[2].scatter(retrieved_misses, [0]*len(retrieved_misses), color="#f59e0b",
                s=120, marker="x", label="Missed correct turn", zorder=3)
axes[2].set_xlabel("Fact Planted at Turn #")
axes[2].set_ylabel("Correct Turn Retrieved?")
axes[2].set_title("Vector Store: Did It Retrieve the Right Turn?")
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["No", "Yes"])
axes[2].legend(loc="center right")

plt.tight_layout()
plt.savefig("vector_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nVector Store Memory: {scores[0]}/{len(RECALL_QUESTIONS)} recalled")
print(f"Sliding Window Memory: {scores[1]}/{len(RECALL_QUESTIONS)} recalled")

### 检索质量分析

我们再深入看看，向量库为每个召回问题检索到了*什么*。即使答案正确，看看哪些轮次被召回、其中有没有「正确」的那一轮，也很有参考价值。

In [ ]:
print("=== Retrieval Analysis ===\n")
print(f"{'Question':<40} {'Planted':>7} {'Retrieved':>25} {'Hit?':>5}")
print("\u2500" * 85)

hits = 0
for r in vec_results:
    planted = r["planted_at"]
    retrieved = r.get("retrieved_turns", [])
    hit = planted in retrieved
    if hit:
        hits += 1
    print(f"{r['question'][:38]:<40} Turn {planted:<3} {str(retrieved):<25} {'\u2713' if hit else '\u2717':>5}")

print(f"\nRetrieval precision: {hits}/{len(vec_results)} queries retrieved the correct source turn.")

## 调优 Top-K：召回与 token 成本

选对 *K* 是经典的查准率与查全率权衡：
- **K 小**（1-2）：提示词里 token 更少，但可能漏掉相关上下文。
- **K 大**（10 以上）：更容易包含正确的记忆，但增加 token 成本，也可能引入无关噪音。

我们用同样的 50 轮对话，测量检索命中率随 K 的变化。

In [ ]:
# 用不同的 K 值测试已存储的向量
k_values = [1, 2, 3, 5, 8, 10, 15]
k_hit_rates = []

for k in k_values:
    hits = 0
    for question, keyword, planted_at in RECALL_QUESTIONS:
        retrieved = vec_mem.search(question, top_k=k)
        retrieved_turns = [r["turn"] for r in retrieved]
        if planted_at in retrieved_turns:
            hits += 1
    hit_rate = hits / len(RECALL_QUESTIONS)
    k_hit_rates.append(hit_rate)
    print(f"  K={k:2d}: {hits}/{len(RECALL_QUESTIONS)} hits ({hit_rate:.0%})")

# 画图
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, k_hit_rates, "o-", color="#4f46e5", linewidth=2, markersize=8)
ax.fill_between(k_values, k_hit_rates, alpha=0.1, color="#4f46e5")
ax.set_xlabel("Top-K")
ax.set_ylabel("Retrieval Hit Rate")
ax.set_title("Retrieval Hit Rate vs. Top-K")
ax.set_ylim(0, 1.05)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("topk_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

我们比较三种记忆策略的 token 成本增长。全量缓冲每次调用都发送所有消息，成本线性增长。滑动窗口成本平稳，但会忘掉旧上下文。向量库成本平稳，还能访问全部历史轮次。

In [ ]:
# 比较各方案的理论 token 成本

MSG_TOKENS = 50   # 每条消息的平均 token 数
SYS_TOKENS = 30   # 系统提示词
K_RETRIEVE = 5    # 检索的 top-K 条记忆
NUM_TURNS = 50

full_buffer, sliding_window, vector_store = [], [], []

for turn in range(1, NUM_TURNS + 1):
    n_msgs = turn * 2

    # 全量缓冲：所有消息
    full_buffer.append(SYS_TOKENS + n_msgs * MSG_TOKENS)

    # 滑动窗口（K=10 条消息）
    sliding_window.append(SYS_TOKENS + min(n_msgs, 10) * MSG_TOKENS)

    # 向量库：K 条检索到的记忆 + 小近期缓冲（4 条消息）
    retrieved_tokens = K_RETRIEVE * 2 * MSG_TOKENS  # 每条记忆包含用户+助手两条消息
    recent_tokens = min(n_msgs, 4) * MSG_TOKENS
    vector_store.append(SYS_TOKENS + retrieved_tokens + recent_tokens)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, full_buffer, "o-", color="#ef4444", label="Full Buffer", linewidth=2, markersize=3)
ax.plot(turns, sliding_window, "s-", color="#f59e0b", label="Sliding Window (K=10)", linewidth=2, markersize=3)
ax.plot(turns, vector_store, "^-", color="#4f46e5", label="Vector Store (top-5 + buffer-4)", linewidth=2, markersize=3)

ax.set_xlabel("Conversation Turn")
ax.set_ylabel("Input Tokens per LLM Call")
ax.set_title("Token Cost per Turn: Three Memory Strategies (50 turns)")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("token_cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"At turn 50:")
print(f"  Full buffer:    {full_buffer[-1]:,} tokens/call")
print(f"  Sliding window: {sliding_window[-1]:,} tokens/call")
print(f"  Vector store:   {vector_store[-1]:,} tokens/call")
print(f"\nVector store keeps cost constant while maintaining full recall.")

## 讨论与权衡

### 优点
- **语义召回**：按*语义*而不是位置检索上下文。第 3 轮的事实，在第 50 轮和第 4 轮一样容易取回。
- **token 成本有界**：每轮成本大致恒定，即 `top_K x chunk_size + recent_buffer`，不随对话长度变化。
- **可扩展**：向量数据库能处理百万级向量。配合 ANN 索引，记忆可以持续增长而不拖慢检索速度。
- **可组合**：能自然配合元数据过滤（时间范围、主题、用户 ID），做更有针对性的检索。

### 缺点
- **基础设施开销**：需要一个嵌入模型和一个向量数据库，每一轮都增加复杂度、延迟和成本。
- **嵌入质量瓶颈**：如果嵌入模型不能很好地刻画你所在领域的语义，检索就不可靠。
- **没有叙事连续性**：检索到的记忆是孤立片段，不是连续的故事。LLM 必须从散落的碎片里自己拼出连贯性。
- **语义相似不等于相关**：两段话可能语义相似但上下文无关，比如「I like Python」和「Python is a snake」。
- **冷启动**：存储的记忆很少时，检索结果噪音大。记忆库变大后，效果会改善。

### 什么时候用向量库记忆

| 场景 | 建议 |
|----------|---------------|
| 长对话（50 轮以上），用户会回头提起早前的话题 | 合适。这是理想用例。 |
| 需要记住之前对话的多会话智能体 | 合适。跨会话做嵌入。 |
| 每一毫秒都很关键的实时聊天 | 谨慎。嵌入会增加约 50-100ms 延迟。 |
| 短对话（10 轮以内） | 大材小用。滑动窗口就够。 |
| 需要严格按时间顺序的叙事 | 不理想。改用缓冲或摘要。 |
| 混合需求：召回旧事实 + 理解近期脉络 | 合适。把向量库和近期缓冲组合使用（本 notebook 就是这么做的）。 |

### 成本模型
每一轮：
- **1 次嵌入调用**处理用户查询（`text-embedding-3-small` 约 $0.00002）
- **1 次嵌入调用**存储这轮对话（约 $0.00002）
- **1 次向量搜索**（ChromaDB 内存模式免费，约 10ms）
- **1 次 LLM 调用**，消耗 `top_K x chunk_size + buffer_size` 个 token

和 LLM 调用相比，嵌入成本可以忽略。真正的省钱之处在于*不用*发送整段对话历史。

## 延伸阅读

- [ChromaDB 文档](https://docs.trychroma.com/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：本 notebook 使用的内存向量数据库
- [OpenAI 嵌入指南](https://platform.openai.com/docs/guides/embeddings?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：text-embedding-3-small 的原理与定价
- [LangChain VectorStoreRetrieverMemory](https://python.langchain.com/docs/modules/memory/types/vectorstore_retriever_memory?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：框架层面的集成
- [Pinecone：面向生产环境的向量数据库](https://www.pinecone.io/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：托管向量数据库替代方案
- [FAISS：Facebook AI Similarity Search](https://github.com/facebookresearch/faiss)：高性能本地向量搜索
- [Anthropic：Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：多轮对话模式
- [Lilian Weng，"LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：记忆部分讲了向量检索模式

---

*← 上一篇：[05：token 缓冲记忆](../05_token_buffer_memory/) · 下一篇：[07：实体记忆](../07_entity_memory/) →*

In [ ]:
# 清理演示过程中生成的临时文件
import os
for f in ["vector_vs_window.png", "topk_tuning.png", "token_cost_comparison.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 动手练习

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：嵌入模型对比
把 `_embed()` 里的 `text-embedding-3-small` 换成 `text-embedding-3-large`，重跑同样的 50 轮实验，比较 K=5 时的命中率。记录每个模型的嵌入延迟，看看速度与质量之间的权衡。

### 挑战 2：K 参数扫描
分别用 K=1、3、5、10、20 做检索。对每个 K，用召回评估测出事实保留率，画出 K 与召回率、K 与每轮平均输入 token 的关系图，找出每花一个 token 能最大化召回的那个 K。

### 挑战 3：带元数据过滤的检索
给每条存储的对话加上 `turn_number` 元数据字段，实现一个按新近度过滤的 `_retrieve()` 变体（向量搜索只考虑最近 N 轮）。参考技术 18 时序记忆（Temporal Memory）的思路，比较这种带时间边界的向量搜索和纯语义检索的效果。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--06-vector-store-memory--vector-store-memory)
